# 0000 - Env Check (prueba inicial)

**Objetivo:** verificar que el setup del pod funciona sin correr inferencia ni nada pesado.

Si todas las celdas pasan en verde, tu entorno está listo para los siguientes notebooks.

**Kernel:** `Python 3.11 (FutBot CPU)` — selecciónalo arriba a la derecha si no está activo.

**Tiempo estimado:** 1 minuto.

## 1. Versión de Python + ubicación del venv

In [ ]:
import sys
print('Python:', sys.version.split()[0])
print('Executable:', sys.executable)

assert sys.version_info >= (3, 11), 'Se requiere Python 3.11+'
assert '/workspace/envs/futbot-cpu' in sys.executable, 'venv no activo — selecciona kernel "Python 3.11 (FutBot CPU)"'
print('OK: venv activo')

## 2. Librerías clave del stack SAM 3

In [ ]:
import importlib

LIBS = {
    'torch': '2.0',
    'torchvision': '0.15',
    'transformers': '4.0',
    'accelerate': '0.20',
    'peft': '0.10',
    'huggingface_hub': '0.20',
    'safetensors': '0.4',
    'cv2': '4.0',
    'decord': '0.6',
    'av': '12.0',
    'numpy': '1.26',
    'pandas': '2.0',
    'matplotlib': '3.7',
    'PIL': '10.0',
    'tqdm': '4.0',
    'einops': '0.7',
    'supervision': '0.20',
    'omegaconf': '2.0',
    'timm': '1.0',
}

failed = []
for lib, min_ver in LIBS.items():
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  ✓ {lib:20s} {ver}')
    except ImportError as e:
        print(f'  ✗ {lib:20s} MISSING — {e}')
        failed.append(lib)

if failed:
    raise RuntimeError(f'Librerías faltantes: {failed}')
print('\nOK: todas las librerías importan correctamente')

## 3. Detección GPU (solo informativo)

En el CPU pod esto debe decir `False` — es lo esperado. Cuando alguien prenda el GPU pod (5090) y monte el mismo volume, esta celda dirá `True`.

In [ ]:
import torch

print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print('CUDA version:', torch.version.cuda)
else:
    print('Estás en el CPU pod (esperado para edición). Usa GPU pod para training.')

## 4. Paths del proyecto

Verifica que las rutas críticas existen.

In [ ]:
from pathlib import Path

WORKSPACE = Path('/workspace')
REPO = WORKSPACE / 'FutBotMX-UAQTeam'
DATA = REPO / 'data' / 'raw'          # symlink a /workspace/Meta_Glasses
ASSETS = REPO / 'assets'
SAM3 = ASSETS / 'sam3'
ENVS = WORKSPACE / 'envs' / 'futbot-cpu'

paths = {
    'workspace': WORKSPACE,
    'repo':      REPO,
    'data_raw':  DATA,
    'assets':    ASSETS,
    'sam3':      SAM3,
    'venv':      ENVS,
}

for name, p in paths.items():
    status = 'OK' if p.exists() else 'MISSING'
    target = f' → {p.resolve()}' if p.is_symlink() else ''
    print(f'  [{status:7s}] {name:10s} {p}{target}')

## 5. SAM 3 — archivos del modelo descargado

Verifica que los archivos del checkpoint están presentes y tienen tamaño esperado.

In [ ]:
EXPECTED = {
    'config.json':            (10_000,         100_000),
    'model.safetensors':      (3_000_000_000,  3_700_000_000),
    'tokenizer.json':         (1_000_000,      10_000_000),
    'tokenizer_config.json':  (100,            5_000),
    'processor_config.json':  (100,            5_000),
    'vocab.json':             (500_000,        2_000_000),
    'merges.txt':             (100_000,        1_000_000),
    'special_tokens_map.json':(100,            5_000),
}

missing = []
for fname, (min_b, max_b) in EXPECTED.items():
    fp = SAM3 / fname
    if not fp.exists():
        print(f'  ✗ {fname:25s} MISSING')
        missing.append(fname)
        continue
    size = fp.stat().st_size
    size_h = f'{size/1e6:.1f} MB' if size > 1e6 else f'{size:,} B'
    ok = '✓' if min_b <= size <= max_b else '⚠'
    print(f'  {ok} {fname:25s} {size_h}')

if missing:
    raise RuntimeError(f'Faltan archivos SAM 3: {missing}. Vuelve a correr setup_hf_sam3.sh')
print('\nOK: SAM 3 checkpoint completo')

## 6. Dataset Meta_Glasses — inventario

Lista los videos disponibles en `data/raw`. Cuenta total, tamaño total, breakdown por subfolder.

**Nota:** Si la descarga del Drive sigue en curso, los números aquí no serán los finales (123 archivos / ~15 GB esperados).

In [ ]:
import os

VIDEOS = list(DATA.rglob('*.mov')) + list(DATA.rglob('*.MOV')) + list(DATA.rglob('*.mp4')) + list(DATA.rglob('*.MP4'))

if not VIDEOS:
    print(f'No hay videos aún en {DATA}.')
    print('Estado actual de la descarga: revisa /workspace/rclone.log o /workspace/gdown.log')
else:
    total_size = sum(v.stat().st_size for v in VIDEOS)
    print(f'Total videos: {len(VIDEOS)}')
    print(f'Total size:   {total_size/1e9:.2f} GB')
    print(f'Esperado:     123 archivos / 15.23 GB\n')

    # Breakdown por carpeta inmediata bajo DATA
    from collections import Counter
    folder_counts = Counter()
    folder_sizes  = Counter()
    for v in VIDEOS:
        rel = v.relative_to(DATA)
        top = rel.parts[0] if len(rel.parts) > 1 else '(root)'
        folder_counts[top] += 1
        folder_sizes[top]  += v.stat().st_size

    print(f'{"Carpeta":30s}  {"Archivos":>10s}  {"Tamaño":>10s}')
    print('-' * 55)
    for folder, count in folder_counts.most_common():
        size_gb = folder_sizes[folder] / 1e9
        print(f'{folder:30s}  {count:>10d}  {size_gb:>8.2f} GB')

## 7. Espacio en disco del volume

In [ ]:
import shutil
total, used, free = shutil.disk_usage('/workspace')
print(f'Total:   {total/1e9:7.1f} GB (capacidad host — no es el volume real)')
print(f'Used:    {used/1e9:7.1f} GB')
print(f'Free:    {free/1e9:7.1f} GB\n')

# Volume real: nuestro plan tiene 60 GB
import subprocess
out = subprocess.run(['du', '-sh', '/workspace'], capture_output=True, text=True)
print('du /workspace:', out.stdout.strip())

# Breakdown
for sub in ['Meta_Glasses', 'FutBotMX-UAQTeam', 'envs', '.pip-cache', '.cache']:
    p = WORKSPACE / sub
    if p.exists():
        out = subprocess.run(['du', '-sh', str(p)], capture_output=True, text=True)
        print(' ', out.stdout.strip())

## 8. HF token + git config

Solo confirma que están seteados, no muestra valores sensibles.

In [ ]:
import os

# Cargar /workspace/.env si existe
env_file = Path('/workspace/.env')
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

checks = {
    'HF_TOKEN':                 'HF_TOKEN' in os.environ,
    'HUGGING_FACE_HUB_TOKEN':   'HUGGING_FACE_HUB_TOKEN' in os.environ,
    'HF_HUB_CACHE':             'HF_HUB_CACHE' in os.environ,
    '/workspace/.env existe':   env_file.exists(),
}
for name, ok in checks.items():
    print(f'  {"✓" if ok else "✗"} {name}')

# git config
import subprocess
print('\ngit config (en repo):')
for k in ['user.name', 'user.email']:
    out = subprocess.run(['git', '-C', str(REPO), 'config', '--get', k], capture_output=True, text=True)
    val = out.stdout.strip() or '(no seteado — corre: git config user.name "Tu Nombre")'
    print(f'  {k}: {val}')

## ✅ Done

Si llegaste hasta aquí sin errores, tu entorno está listo.

**Siguiente paso:** abre `01_supervision_yolo.ipynb` (cuando lo creemos) para empezar con vision pipeline básico.

**Si algo falló:** comparte el output en el chat del equipo. Lo más común es:
- Kernel incorrecto → selecciona `Python 3.11 (FutBot CPU)` arriba a la derecha
- Archivo SAM 3 faltante → corre desde terminal: `bash /tmp/setup_hf_sam3.sh`
- Symlink data faltante → corre desde terminal: `ln -sfn /workspace/Meta_Glasses /workspace/FutBotMX-UAQTeam/data/raw`